# Regressão Logística — contrato, custo de ajuste e espaço de busca

Este notebook é a frente de **modelos interpretáveis** da comparação da Seção 4.4 (ART.7). Ele
começa onde o `modelagem.ipynb` da Seção 4.3 termina: consome o contrato de dados já pronto e não
repete nenhuma etapa de limpeza, integração ou pré-processamento.

Princípios adotados, os mesmos do notebook da Seção 4.3:

- consumir a matriz do contrato, sem reprocessar as fontes brutas nem reajustar transformador;
- importar de `src/` em vez de colar código na célula, para que notebook e `pytest` exercitem
  exatamente o mesmo módulo;
- nenhuma partição criada aqui. A divisão por data e por Cliente já veio congelada, e recriá-la
  neste notebook produziria uma segunda divisão que ninguém acordou.

**Seção 1, entregue pelo card #206.** Mede quanto custa um ajuste da Regressão Logística sobre o
contrato e se ela converge. Os dois números existem para dimensionar a grade de `GridSearchCV`
do card #207 antes de a busca começar, e não para avaliar qualidade: nenhuma métrica de
desempenho sai desta seção.

## 1. Custo de um ajuste e convergência (card #206)

### 1.1. Dependências

As versões de `scikit-learn` e `scipy` estão fixas em `requirements.txt` a partir do Sprint 04.
A fixação não é burocracia: o número de iterações até convergir, medido na Seção 1.4, depende da
implementação do solver, e um piso `>=` deixaria o Colab instalar outra minor e devolver um custo
diferente do que a grade do card #207 assume.

In [ ]:
!pip install -q -r requirements.txt

import sys
import time
from pathlib import Path

import pandas as pd
import scipy
import sklearn

# Inclui src/ no caminho de importação tanto no projeto local quanto no Colab.
# O break evita que duas cópias dos módulos fiquem no caminho ao mesmo tempo, o
# que tornaria ambíguo qual delas o import resolve.
for pasta_codigo in [Path("src"), Path("../src"), Path("/content/src")]:
    if pasta_codigo.exists():
        sys.path.insert(0, str(pasta_codigo.resolve()))
        break

from fumaca_logistica import medir_ajuste_unico, menor_max_iter_que_converge
from matriz import preparar_matriz, resumo_da_matriz

print("scikit-learn", sklearn.__version__)
print("scipy", scipy.__version__)

As versões impressas têm que ser `scikit-learn 1.9.1` e `scipy 1.18.1`, as mesmas declaradas em
`requirements.txt`. Divergência aqui invalida a comparação de custo das seções seguintes: o tempo
medido passaria a descrever outra implementação do `lbfgs`, e a conta que dimensiona a grade do
card #207 deixaria de valer.

### 1.2. Carga da base analítica

A busca cobre o projeto local e o Colab, com e sem Drive montado. Se nenhum caminho existir, a
célula falha com a instrução do que fazer, em vez de seguir com uma base vazia. A base não é
versionada, conforme o Termo de Abertura, que veda publicar dados do parceiro.

In [ ]:
# No Colab, monte o Drive antes de rodar esta célula:
#   from google.colab import drive; drive.mount('/content/drive')

CAMINHOS_POSSIVEIS = [
    Path("data/processed/base_analitica.parquet"),
    Path("../data/processed/base_analitica.parquet"),
    Path("/content/data/processed/base_analitica.parquet"),
    Path("/content/drive/MyDrive/projeto-azul/data/processed/base_analitica.parquet"),
]

caminho_base = next((c for c in CAMINHOS_POSSIVEIS if c.exists()), None)
if caminho_base is None:
    raise FileNotFoundError(
        "base_analitica.parquet não encontrada. Rode notebooks/pre-processamento.ipynb "
        f"ou ajuste o caminho. Procurado em: {[str(c) for c in CAMINHOS_POSSIVEIS]}"
    )

df = pd.read_parquet(caminho_base)
print(f"{caminho_base}: {df.shape[0]:,} linhas, {df.shape[1]} colunas")
print(f"Clientes distintos: {df['ID_GOLDENRECORD'].nunique():,}")

A base tem **484.915 linhas e 51 colunas**, com **407.139 valores distintos de
`ID_GOLDENRECORD`**. A distância entre os dois números é o motivo de a validação cruzada do card
#210 precisar agrupar por Cliente: há Cliente com mais de uma resposta, e a Hipótese 4 da Seção
4.2.4 mostrou que quem detratou uma vez volta a detratar com chance 4,61 vezes maior. Um Cliente
presente em dois folds faria o modelo reconhecer a pessoa em vez de aprender o fenômeno.

### 1.3. Matriz do contrato

`preparar_matriz`, de `src/matriz.py`, executa numa chamada as três etapas que precisam acontecer
nesta ordem: particionar por data e por Cliente, selecionar a allowlist de features do score
pós-viagem e ajustar o pré-processador **apenas no treino**.

Esta é a única fonte de `X`, `y`, `grupos` e do transformador em todo o notebook. As datas de
corte vêm do registro de decisão do #127, não são escolhidas aqui.

In [ ]:
CORTE_VALIDACAO = "2025-07-01"
CORTE_TESTE = "2026-01-01"

inicio = time.perf_counter()
preparo = preparar_matriz(df, corte_validacao=CORTE_VALIDACAO, corte_teste=CORTE_TESTE)
print(f"preparar_matriz: {time.perf_counter() - inicio:.1f}s")

matriz_treino = preparo["matrizes"]["treino"]
alvo_treino = preparo["y"]["treino"]
grupos_treino = preparo["grupos"]["treino"]

print(f"matriz de treino: {matriz_treino.shape[0]:,} linhas x {matriz_treino.shape[1]} colunas")
print(f"colunas na saída do pré-processador: {len(preparo['preprocessador'].get_feature_names_out())}")
print(f"Clientes no treino: {grupos_treino.nunique():,}")
print(f"prevalência de Detrator no treino: {alvo_treino.mean():.4f}")

resumo_da_matriz(preparo)

O contrato entrega uma matriz de treino com **341.962 linhas e 38 colunas**, cobrindo **307.510
Clientes**, e a prevalência de Detrator no treino é de **0,2043**. As 38 colunas da matriz são
exatamente as 38 que `get_feature_names_out` reporta, o que confirma que nada foi transformado
fora do contrato. O preparo inteiro custa cerca de **5,3 segundos**, desprezível diante do ajuste
medido a seguir.

A prevalência de 0,2043 é o número que o card #207 usa para justificar o eixo `class_weight` da
grade: com quatro não Detratores para cada Detrator, o modelo sem reponderação tende a acertar a
classe majoritária e a errar justamente quem a operação precisa encontrar.

### 1.4. Ajuste único: quanto custa e se converge

`medir_ajuste_unico` ajusta `LogisticRegression` **uma vez** sobre a matriz do contrato e devolve
tempo, iterações e convergência. Ele não monta `Pipeline`, que é o card #208, não reparticiona e
não calcula métrica: o número que interessa aqui é o relógio.

O aviso de convergência é capturado em vez de silenciado. Um ajuste que para no limite de
iterações entrega coeficiente provisório, e o card #212 lê exatamente esses coeficientes como
odds ratio.

In [ ]:
medicao_padrao = medir_ajuste_unico(matriz_treino, alvo_treino)

for chave in ("max_iter", "solver", "n_iter", "convergiu", "segundos"):
    print(f"{chave}: {medicao_padrao[chave]}")

if medicao_padrao["aviso"]:
    print("\naviso de convergência:")
    print(medicao_padrao["aviso"])

**O achado do card: a Regressão Logística não converge no `max_iter` padrão.** Com
`max_iter=100`, `solver='lbfgs'` e semente 42, o ajuste consome as 100 iterações, leva cerca de
**8,0 segundos** e emite `ConvergenceWarning` com `STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT`.

**A escala das features não é a causa, e isso importa.** O aviso do scikit-learn sugere escalonar
os dados, mas o contrato já faz isso: `matriz._montar_preprocessador` aplica `RobustScaler` nas
numéricas e codificação one-hot nas categóricas, então o `lbfgs` já recebe colunas em ordem de
grandeza comparável. Montar um escalonamento novo por fora do contrato para tentar acelerar a
convergência produziria uma segunda matriz e quebraria a comparação da Seção 4.4, que exige que
os quatro modelos consumam a mesma. O caminho correto é subir o `max_iter`, que é o que a Seção
1.5 mede.

A consequência para o card #207 é direta: `max_iter` não pode ficar no default na grade de busca,
senão toda combinação testada seria avaliada com coeficiente truncado.

### 1.5. Qual `max_iter` basta, e quanto ele custa

`menor_max_iter_que_converge` sobe o limite pela escala 100, 200, 400, 800, 1600 e para na
primeira tentativa que converge. Dobrar a cada passo mantém o custo total de procurar da ordem do
último ajuste, em vez da soma de uma varredura fina.

A trilha de tentativas é registrada de propósito: **só o tempo do ajuste que converge entra na
conta de custo da grade do card #207.** Os ajustes truncados param antes e custam menos, e usá-los
subestimaria a busca.

In [ ]:
busca_max_iter = menor_max_iter_que_converge(matriz_treino, alvo_treino)

for tentativa in busca_max_iter["tentativas"]:
    estado = "converge" if tentativa["convergiu"] else "trunca"
    print(f"max_iter={tentativa['max_iter']:>5} | n_iter={tentativa['n_iter']:>5} | "
          f"{tentativa['segundos']:>6.1f}s | {estado}")

escolhido = busca_max_iter["escolhido"]
print(f"\nescolhido: max_iter={escolhido['max_iter']}, "
      f"converge em {escolhido['n_iter']} iterações, {escolhido['segundos']:.1f}s")

A escala mostra onde está o limite:

| `max_iter` | iterações usadas | tempo | resultado |
|---:|---:|---:|---|
| 100 | 100 | 9,2s | trunca |
| 200 | 200 | 18,2s | trunca |
| 400 | 400 | 36,5s | trunca |
| 800 | **534** | **47,4s** | **converge** |

**A Regressão Logística converge em 534 iterações e custa cerca de 47 segundos por ajuste**, medido
nesta máquina (WSL2 local, CPU, scikit-learn 1.9.1 e scipy 1.18.1). No Colab o número muda com a
máquina sorteada, e por isso o que se transporta para o card #207 não é o segundo exato e sim a
ordem de grandeza: **um ajuste que converge custa dezenas de segundos, não centenas de
milissegundos.**

Até 400 iterações o tempo cresce proporcionalmente ao limite, porque o ajuste é truncado e gasta
todas as iterações disponíveis. Em 800 ele para antes, em 534, e é por isso que o tempo não dobra
de novo.

**O que este card entrega ao card #207**, que é quem declara a grade:

- `max_iter` precisa entrar fixado em pelo menos **800** na grade, e não como default;
- a conta de custo da busca é **47s por ajuste x número de folds x número de combinações**. Com os
  5 folds da validação por Cliente, cada combinação custa cerca de 4 minutos em série, o que
  limita a grade a poucas dezenas de combinações dentro de uma sessão do Colab;
- a prevalência de **0,2043** é o dado que sustenta o eixo `class_weight` da grade.

O mesmo número vai para a Luiza, que precisa dele para dimensionar a grade da Árvore de Decisão
sob o mesmo critério.